# CasTuner ODE steps — tiny subset demo (Steps 2/3 + GOF)

This notebook uses the **parameter CSVs** (from Step 1a/1b/1c) and demonstrates:
- ODE simulation for **derepression (REV)** and **repression (KD)**
- A minimal “goodness-of-fit” check using MAE / R² style comparisons

It runs on a small subset and synthetic observations (so you can validate the mechanics).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

from pathlib import Path

PARAM_PATH = Path("parameters")


## 1) Load parameters (or create defaults if missing)


In [ ]:
def safe_read(path, default_df):
    if path.exists():
        return pd.read_csv(path)
    return default_df

up = safe_read(PARAM_PATH/"half_times_upregulation.csv",
               pd.DataFrame([{"plasmid":"SP411","t_up":6.0},{"plasmid":"SP430A","t_up":9.0}]))
down = safe_read(PARAM_PATH/"half_times_downregulation.csv",
                 pd.DataFrame([{"plasmid":"SP411","t_down":7.0},{"plasmid":"SP430A","t_down":11.0}]))
hill = safe_read(PARAM_PATH/"Hill_parameters.csv",
                 pd.DataFrame([{"plasmid":"SP411","K":0.4,"n":2.0},{"plasmid":"SP430A","K":0.6,"n":1.6}]))

# alpha is global in your scripts; make a default if missing
alpha_path = PARAM_PATH/"alphamcherry.csv"
if alpha_path.exists():
    alpha_df = pd.read_csv(alpha_path)
    alpha = float(alpha_df["alpha"].iloc[0])
else:
    alpha = 0.08  # 1/h
alpha


## 2) Define ODEs (matching the scripts)

Your pipeline uses (simplified form):

**KD (repression)**:
- `beta = ln(2)/t_up`
- `dR/dt = beta - R*(ln2/t_up)`
- `dY/dt = K^n/(K^n + R^n) - alpha*Y`

**REV (derepression)** is analogous but uses `t_down` in the R-dynamics.

We'll simulate both with the same Y-equation form.


In [ ]:
import math

def rhs_kd(t, y, t_up, K, n, alpha):
    R, Y = y
    t_up = max(float(t_up), 1e-6)
    K = max(float(K), 1e-12)
    n = max(float(n), 1e-6)
    alpha = max(float(alpha), 1e-12)

    beta = math.log(2.0) / t_up
    Rpos = max(R, 0.0)
    Kn = K**n
    Rn = (Rpos**n)
    dR = beta - Rpos * (math.log(2.0) / t_up)
    dY = (Kn / (Kn + Rn)) - alpha * Y
    return [dR, dY]

def rhs_rev(t, y, t_down, K, n, alpha):
    # Simple mirror: R decays with half-time t_down toward 0
    R, Y = y
    t_down = max(float(t_down), 1e-6)
    K = max(float(K), 1e-12)
    n = max(float(n), 1e-6)
    alpha = max(float(alpha), 1e-12)

    # R decays: dR = -R * ln2 / t_down
    Rpos = max(R, 0.0)
    dR = -Rpos * (math.log(2.0) / t_down)

    Kn = K**n
    Rn = (Rpos**n)
    dY = (Kn / (Kn + Rn)) - alpha * Y
    return [dR, dY]

def simulate(fun, y0, t_end=48.0, dt=0.1, **kwargs):
    t_eval = np.arange(0.0, t_end + dt/2, dt)
    sol = solve_ivp(lambda t, y: fun(t, y, **kwargs),
                    t_span=(0.0, t_end), y0=y0, t_eval=t_eval, method="LSODA")
    if not sol.success:
        raise RuntimeError(sol.message)
    return pd.DataFrame({"time_h": sol.t, "R": sol.y[0], "Y": sol.y[1]})


## 3) Simulate two constructs


In [ ]:
plasmids = ["SP411", "SP430A"]
sim = {}

for pl in plasmids:
    t_up = float(up.loc[up["plasmid"]==pl, "t_up"].iloc[0])
    t_down = float(down.loc[down["plasmid"]==pl, "t_down"].iloc[0])
    K = float(hill.loc[hill["plasmid"]==pl, "K"].iloc[0])
    n = float(hill.loc[hill["plasmid"]==pl, "n"].iloc[0])

    # Start conditions: KD starts with low R (0), REV starts with high R (~1)
    y0_kd = [0.0, 1.0/max(alpha,1e-12)]
    y0_rev = [1.0, 1.0/max(alpha,1e-12)]

    sim[(pl,"KD")] = simulate(rhs_kd, y0_kd, t_end=48, dt=0.1, t_up=t_up, K=K, n=n, alpha=alpha)
    sim[(pl,"Rev")] = simulate(rhs_rev, y0_rev, t_end=48, dt=0.1, t_down=t_down, K=K, n=n, alpha=alpha)

list(sim.keys()), sim[("SP411","KD")].head()


## 4) Make “synthetic observations” and compute a tiny GOF score

The real pipeline compares simulated `alpha*Y` vs experimental fold-change of mCherry.
We'll imitate that: sample a few time points, add noise, compute MAE.


In [ ]:
rng = np.random.default_rng(12)

obs_rows = []
for (pl, exp), df in sim.items():
    # Observation time points (subset)
    t_obs = np.array([0, 2, 4, 8, 12, 24, 36, 48], float)
    y = np.interp(t_obs, df["time_h"], (df["Y"]*alpha))  # alpha*Y
    y_noisy = y + rng.normal(0, 0.03, size=y.size)
    for t, yy in zip(t_obs, y_noisy):
        obs_rows.append({"plasmid": pl, "exp": exp, "time_h": t, "fc_cherry": float(yy)})
obs = pd.DataFrame(obs_rows)

def mae(a,b):
    a = np.asarray(a,float); b=np.asarray(b,float)
    m = np.isfinite(a)&np.isfinite(b)
    if m.sum()==0: return np.nan
    return float(np.mean(np.abs(a[m]-b[m])))

gof = []
for (pl, exp), sub in obs.groupby(["plasmid","exp"]):
    df = sim[(pl, exp)]
    pred = np.interp(sub["time_h"], df["time_h"], df["Y"]*alpha)
    gof.append({"plasmid":pl, "exp":exp, "MAE": mae(sub["fc_cherry"], pred)})
pd.DataFrame(gof)


## 5) Plot: data vs simulation (subset)


In [ ]:
for pl in plasmids:
    fig, ax = plt.subplots(figsize=(7,4))
    for exp in ["KD","Rev"]:
        df = sim[(pl, exp)]
        ax.plot(df["time_h"], df["Y"]*alpha, label=f"{exp} sim")
        sub = obs.query("plasmid==@pl and exp==@exp")
        ax.scatter(sub["time_h"], sub["fc_cherry"], label=f"{exp} obs", alpha=0.8)
    ax.set_title(f"{pl} — alpha*Y vs time (toy subset)")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("mCherry fold-change (proxy)")
    ax.legend()
    plt.show()


## Next: switch from synthetic observations to your real tables

Replace the `obs` DataFrame with the pipeline’s real per-time means:

- REV: outputs from `step_2_simulate_derepression.py`
- KD:  outputs from `step_3_simulate_repression.py`

You can also import those scripts and call their functions (see your `step_6_goodness_of_fit.py`).
